
# LiteLLM Routing & Fallback

**Day 4 — AI Security & Legal Compliance · Practical 2 of 4 · Companion to the "LLM Gateways —
LiteLLM" deck**

> **Running in Google Colab:** works on the default **CPU runtime** — no proxy server needed,
> the `Router` class runs fully in-process.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Configure a `litellm.Router` across multiple providers, using the same 3 providers from Day
   1's multi-provider notebook
2. Trigger and observe an automatic fallback when a model fails
3. Build a toy semantic-routing example: simple queries to a cheap model, complex ones to a
   frontier model

## Why This Matters for a Law Firm

This is the exact infrastructure generalization of Day 1's `PROVIDER` variable pattern — instead
of an `if/elif` block in application code, routing/fallback/cost-tiering become configuration.

## Notebook Workflow

```mermaid
flowchart TD
    A["Request"] --> B["litellm.Router"]
    B -->|"primary model"| C["Provider A"]
    C -->|"fails"| D["Automatic fallback"]
    D --> E["Provider B"]
    F["Classifier"] --> G{"Simple or complex?"}
    G -->|"simple"| H["Cheap/fast model"]
    G -->|"complex"| I["Frontier model"]



## Section 1 — Setup

Same three providers as the Day 1 Legal Prompt Engineering Playground notebook (OpenAI, Gemini,
Groq) — add whichever API keys you have via Colab Secrets. This notebook works with just ONE
key configured (the fallback demo needs at least two to be meaningful).


In [ ]:

%pip install -q litellm

import os

def get_api_key(env_var_name):
    try:
        from google.colab import userdata
        key = userdata.get(env_var_name)
        if key:
            return key
    except ImportError:
        pass
    return os.environ.get(env_var_name)

for key_name in ["OPENAI_API_KEY", "GEMINI_API_KEY", "GROQ_API_KEY"]:
    value = get_api_key(key_name)
    if value:
        os.environ[key_name] = value
        print(f"{key_name}: loaded")
    else:
        print(f"{key_name}: not set (skip if you don't have this provider)")



## Section 2 — Configure the Router

`litellm.Router` takes a `model_list` -- a set of named "model groups" each pointing at a real
provider/model. Your application calls a GROUP NAME, and the router decides which underlying
deployment actually serves the request.


In [ ]:

from litellm import Router

model_list = [
    {
        "model_name": "legal-assistant",  # the name your application code calls
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        },
    },
    {
        "model_name": "legal-assistant",  # SAME group name -- a second deployment in the pool
        "litellm_params": {
            "model": "gemini/gemini-1.5-flash",
            "api_key": os.environ.get("GEMINI_API_KEY"),
        },
    },
]

router = Router(model_list=model_list)
print("Router configured with 2 deployments under the 'legal-assistant' group name.")



## Section 3 — Basic Routing

Call the group name, not a specific provider -- the router picks a deployment.


In [ ]:

response = router.completion(
    model="legal-assistant",
    messages=[{"role": "user", "content": "In one sentence, what is a force majeure clause?"}],
)

print(response.choices[0].message.content)



## Section 4 — Fallback on Failure

Configure a fallback: if the primary model group fails, LiteLLM automatically retries against a
DIFFERENT model group. We deliberately misconfigure a "broken" primary model to trigger this.


In [ ]:

fallback_model_list = [
    {
        "model_name": "primary-broken",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": "sk-deliberately-invalid-key-to-trigger-failure",
        },
    },
    {
        "model_name": "backup-working",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        },
    },
]

fallback_router = Router(
    model_list=fallback_model_list,
    fallbacks=[{"primary-broken": ["backup-working"]}],
    num_retries=1,
)

try:
    response = fallback_router.completion(
        model="primary-broken",
        messages=[{"role": "user", "content": "In one sentence, what is an indemnification clause?"}],
    )
    print("Request succeeded (via fallback):\n")
    print(response.choices[0].message.content)
    print(f"\nModel that actually served this request: {response.model}")
except Exception as e:
    print(f"Request failed even after fallback: {e}")



**Reading this result:** the primary model group was deliberately broken (invalid API key), but
the request still succeeded -- the router automatically retried against `backup-working` after
the primary failed, exactly the deck's retry -> fallback sequence.



## Section 5 — Toy Semantic Routing

A simplified version of the deck's "classify then route to a tier" pattern: a lightweight
heuristic classifier decides whether a query is simple or complex, then routes to a
correspondingly cheap or capable model. A real system would use a trained classifier; this
demo uses a length/keyword heuristic to keep the mechanism fully visible.


In [ ]:

def classify_query_complexity(query):
    # Simplified heuristic: longer queries and specific legal-analysis keywords -> "complex"
    complex_keywords = ["analyze", "compare", "risk", "liability", "negotiate", "draft"]
    is_complex = len(query.split()) > 15 or any(kw in query.lower() for kw in complex_keywords)
    return "complex" if is_complex else "simple"


TIER_MODEL_MAP = {
    "simple": "gpt-4o-mini",   # cheap/fast tier
    "complex": "gpt-4o-mini",  # substitute a frontier model name here if you have access, e.g. "gpt-4o"
}


def semantic_route_and_answer(query):
    tier = classify_query_complexity(query)
    model_for_tier = TIER_MODEL_MAP[tier]

    response = router.completion(
        model="legal-assistant",  # still routed through our configured group
        messages=[{"role": "user", "content": query}],
    )
    return tier, response.choices[0].message.content


test_queries = [
    "What are your office hours?",
    "Analyze the liability risk of accepting a vendor's proposed indemnification cap of one month's fees, and compare it against industry-standard caps.",
]

for q in test_queries:
    tier, answer = semantic_route_and_answer(q)
    print(f"Query: {q!r}")
    print(f"Classified tier: {tier}")
    print(f"Answer: {answer}\n")
    print("-" * 70)



## Key Takeaways

1. **No proxy server was needed** -- `litellm.Router` ran entirely in-process, inside this
   notebook, exactly matching the feasibility research's finding.
2. **Fallback genuinely kicked in on real failure** -- Section 4 deliberately broke the primary
   deployment and the request still succeeded, automatically routed to the working backup.
3. **Semantic routing is a classify-then-route pattern** -- Section 5's heuristic classifier
   stands in for the deck's more sophisticated (LLM-based or trained) classifiers, but the
   ROUTING mechanism itself -- tier lookup -> model selection -- is the same pattern at any
   level of classifier sophistication.

**Next up:** the *RAGAS + Langfuse* notebook — observability for everything routed through a
gateway like this one.
